In [26]:
from sklearn.metrics import accuracy_score, matthews_corrcoef, classification_report
from sklearn.preprocessing import label_binarize, LabelEncoder
import json
from pathlib import Path
import pandas as pd

from model.visualization import plot_confusion_matrix
from pymmseqs.commands import createdb, search

# Homology-based Inference (HBI) baseline
### Non redundancy reduced train set. Test set is the same as from the normal model

In [27]:
train_all = pd.read_csv("train_all_df.csv")
val_set = pd.read_csv("../val_data.csv")
test_set = pd.read_csv("../test_data.csv")

In [49]:
test_set["Protein families"].unique()

array(['CRISP family', 'Venom metalloproteinase (M12B) family',
       'Disintegrin family', 'PDGF/VEGF growth factor family',
       'Scoloptoxin family',
       'Natriuretic, Bradykinin potentiating peptide family',
       'Snaclec family', 'Peptidase S1 family', 'nontox',
       'Actinoporin family', 'Short scorpion toxin superfamily',
       'Long chain scorpion toxin family', 'Three-finger toxin family',
       'other', 'Sea anemone type 3 (BDS) potassium channel toxin family',
       'Long (4 C-C) scorpion toxin superfamily',
       'Cnidaria small cysteine-rich protein (SCRiP) family',
       'Vasopressin/oxytocin family', 'Venom Kunitz-type family',
       'PBP/GOBP family', 'FARP (FMRFamide related peptide) family',
       'Phospholipase family', 'Calycin superfamily',
       'Non-disulfide-bridged peptide (NDBP) superfamily',
       'Sea anemone type 1 potassium channel toxin family',
       'Limacoditoxin family', 'Cationic peptide family',
       'Sea anemone sodium channel

### Label Mismatch
label mismatch because the training data used for the HBI consists of the cluster representatives AND their members. Unlike in the normal train-val-test split, the redundancy reduced (and thus only representatives) data was used.
This results in a need to replace the "new" train data families with "other"

In [48]:
# assume best_all is your DataFrame
q_labels = set(train_all["Protein families"].unique())
t_labels = set(val_set["Protein families"].unique())

only_in_query  = q_labels  - t_labels
only_in_target = t_labels  - q_labels
both            = q_labels & t_labels
sym_diff        = q_labels ^ t_labels  # labels not in both

print("Only in train:")
print(only_in_query)
print("\nOnly in val:")
print(only_in_target)
print("\nIn both:")
print(both)
print("\nSymmetric difference (not in both):")
print(sym_diff)


Only in train:
set()

Only in val:
set()

In both:
{'Phospholipase family', 'MCD family', 'PDGF/VEGF growth factor family', 'nontox', 'Formicidae venom family', 'Snaclec family', 'CRISP family', 'Actinoporin family', 'Scoloptoxin family', 'Venom metalloproteinase (M12B) family', 'Peptidase S1 family', 'other', 'Venom Ptu1-like knottin family', 'Venom Kunitz-type family', 'Vasopressin/oxytocin family', 'Sea anemone type 1 potassium channel toxin family', 'Calycin superfamily', 'FARP (FMRFamide related peptide) family', 'Three-finger toxin family', 'Sea anemone sodium channel inhibitory toxin family', 'Insulin family', 'Neurotoxin family', 'Bradykinin-related peptide family', 'Cnidaria small cysteine-rich protein (SCRiP) family', 'Short scorpion toxin superfamily', 'Disintegrin family', 'Limacoditoxin family', 'Conotoxin family', 'Natriuretic, Bradykinin potentiating peptide family', 'Long chain scorpion toxin family', 'Teretoxin family', 'Long (3 C-C) scorpion toxin superfamily', 'Non-d

In [29]:
# Build a mapping of labels that exist only in train (not in val/test) → "other"
repl_map = {lbl: "other" for lbl in only_in_query}

# Apply to both columns (so your ground‐truth and preds share the same reduced label-space)
train_all["Protein families"]  = train_all["Protein families"].replace(repl_map)

# Sanity check: after this, the two label-sets should be identical
print(set(train_all["Protein families"].unique()) ^ set(val_set["Protein families"].unique()))

set()


In [30]:
train = createdb("train_all_members.fasta", "tmp/train_db")
test = createdb("../val_data.fasta", "tmp/val_db")


-------------------- Running a mmseqs2 command --------------------
✓ Detailed execution log has been saved
✓ Database creation completed successfully
  Results saved to: /Users/selin/PycharmProjects/ToxFam/benchmark/HBI/tmp/train_db

-------------------- Running a mmseqs2 command --------------------
✓ Detailed execution log has been saved
✓ Database creation completed successfully
  Results saved to: /Users/selin/PycharmProjects/ToxFam/benchmark/HBI/tmp/val_db


In [31]:
search_res = search(val.to_path(),
                    train.to_path(),
                    "tmp/search_res",
                    "tmp/tmp",
                    s=9,
                    e="inf",
                    min_seq_id=0.0,
                    max_seqs=100_000)


-------------------- Running a mmseqs2 command --------------------
✓ Detailed execution log has been saved
✓ Search completed successfully
  Results saved to: /Users/selin/PycharmProjects/ToxFam/benchmark/HBI/tmp/search_res


In [32]:
res = search_res.to_pandas()
res

Output is not readable. Executing convertalis command to convert the alignment database to a readable format.

-------------------- Running a mmseqs2 command --------------------
✓ Detailed execution log has been saved
✓ ConvertAlis completed successfully
  Results saved to: /Users/selin/PycharmProjects/ToxFam/benchmark/HBI/tmp/search_res.tsv


,query,target,fident,alnlen,mismatch,gapopen,qstart,qend,tstart,tend,evalue,bits
0,P01502,P59261,0.908,26,2,0,1,26,23,48,4.791000e-10,52
1,P01502,P01501,0.908,26,2,0,1,26,23,48,4.791000e-10,52
2,P01502,P59262,0.908,26,2,0,1,26,23,48,4.791000e-10,52
3,P01502,P68407,0.908,26,2,0,1,26,23,48,4.791000e-10,52
4,P01502,P68408,0.908,26,2,0,1,26,23,48,4.791000e-10,52
...,...,...,...,...,...,...,...,...,...,...,...,...
141033784,Q17577,P60268,1.000,3,0,0,374,376,32,34,2.054000e+06,13
141033785,Q17577,P0C5X5,0.297,20,13,0,431,450,10,29,2.054000e+06,13
141033786,Q17577,P0DTJ8,0.548,9,4,0,361,369,2,10,2.054000e+06,13
141033787,Q17577,P80548,0.693,7,2,0,1,7,1,7,2.054000e+06,13


In [33]:
len(res["query"].unique()) # not all test set samples get a hit? --> nan hbi

8540

In [34]:
best_hits = (
    res
    .loc[res.groupby("query")["evalue"].idxmin()]
    .reset_index(drop=True)
)
best_hits

,query,target,fident,alnlen,mismatch,gapopen,qstart,qend,tstart,tend,evalue,bits
0,A0A023W0V6,A0A023W140,0.528,31,14,0,1,31,1,31,3.657000e-03,34
1,A0A075B6S4,P04430,0.887,95,11,0,1,95,1,95,1.908000e-52,178
2,A0A075B6S6,A0A0C4DH68,0.865,100,13,0,1,100,1,100,1.031000e-53,182
3,A0A075B6T6,A0A0B4J271,0.789,93,19,0,1,93,1,93,1.073000e-43,153
4,A0A078BQP2,E7EAU8,0.332,1034,655,0,2,983,43,1076,4.651000e-163,542
...,...,...,...,...,...,...,...,...,...,...,...,...
8535,X2JCV5,A6MFL0,0.849,517,78,0,1,516,1,517,1.290000e-297,909
8536,X5I9Z2,F2XFS9,0.732,42,11,0,1,42,1,42,3.903000e-14,65
8537,X5IWS1,Q9XZK3,0.458,53,26,0,2,54,2,50,1.373000e-07,47
8538,X5M5N0,Q9JIH7,0.621,382,141,0,316,697,202,574,1.579000e-131,466


In [50]:
# prepare df with ground truth (query_label = test gt)
all_queries = val_set[["identifier", "Protein families"]].rename(columns={"identifier": "query","Protein families": "query_label"})
best_all = all_queries.merge(best_hits, on="query", how="left")[["query","target","evalue", "query_label"]]

# add target labels (target_label = HBI prediction)
train_labels = train_all[["identifier", "Protein families"]].rename(columns={"identifier": "target", "Protein families": "target_label"})
best_all = best_all.merge(train_labels, on="target", how="left")

best_all

,query,target,evalue,query_label,target_label
0,P01502,P59261,4.791000e-10,other,other
1,P84808,Q8JI38,6.068000e-140,CRISP family,CRISP family
2,Q962V9,P0DPU0,4.125000e-10,CRISP family,nontox
3,C1IBY3,F8QQG5,2.025000e-132,CRISP family,CRISP family
4,C8YJ99,C8YJA2,1.733000e-65,CRISP family,CRISP family
...,...,...,...,...,...
8598,P84809,D9U2A2,1.613000e-18,Long (3 C-C) scorpion toxin superfamily,Long (3 C-C) scorpion toxin superfamily
8599,A0A7S8MU86,Q68PG4,1.354000e-03,Long (3 C-C) scorpion toxin superfamily,Long (4 C-C) scorpion toxin superfamily
8600,A0F0C2,A0A7S8MV32,2.944000e-15,Long (3 C-C) scorpion toxin superfamily,Long (3 C-C) scorpion toxin superfamily
8601,Q9BLM1,P56678,9.216000e-12,Long (3 C-C) scorpion toxin superfamily,Long (4 C-C) scorpion toxin superfamily


In [51]:
# assume best_all is your DataFrame
q_labels = set(best_all["query_label"].unique())
t_labels = set(best_all["target_label"].unique())

only_in_query  = q_labels  - t_labels
only_in_target = t_labels  - q_labels
both            = q_labels & t_labels
sym_diff        = q_labels ^ t_labels  # labels not in both

print("Only in test ground truth:")
print(only_in_query)
print("\nOnly in hbi inference:")
print(only_in_target)
print("\nIn both:")
print(both)
print("\nSymmetric difference (not in both):")
print(sym_diff)


Only in test ground truth:
set()

Only in hbi inference:
{nan}

In both:
{'Phospholipase family', 'MCD family', 'PDGF/VEGF growth factor family', 'nontox', 'Formicidae venom family', 'Snaclec family', 'CRISP family', 'Actinoporin family', 'Scoloptoxin family', 'Venom metalloproteinase (M12B) family', 'Peptidase S1 family', 'other', 'Venom Ptu1-like knottin family', 'Venom Kunitz-type family', 'Vasopressin/oxytocin family', 'Sea anemone type 1 potassium channel toxin family', 'Calycin superfamily', 'FARP (FMRFamide related peptide) family', 'Three-finger toxin family', 'Sea anemone sodium channel inhibitory toxin family', 'Insulin family', 'Neurotoxin family', 'Bradykinin-related peptide family', 'Cnidaria small cysteine-rich protein (SCRiP) family', 'Short scorpion toxin superfamily', 'Disintegrin family', 'Limacoditoxin family', 'Conotoxin family', 'Natriuretic, Bradykinin potentiating peptide family', 'Long chain scorpion toxin family', 'Teretoxin family', 'Long (3 C-C) scorpion toxi

In [52]:
# 0) make a copy so we don’t overwrite your original
df = best_all.copy()

# 1) replace NaN predictions with “no hit”
df["target_label"] = df["target_label"].fillna("no hit")

# 2) build your valid class list off the training set + “no hit”
train_classes = list(train_labels["target_label"].unique())
if "no hit" not in train_classes:
    train_classes.append("no hit")
class_list = train_classes      # order defines axes
cls2idx    = {cls: i for i, cls in enumerate(class_list)}

le = LabelEncoder()
le.fit(class_list)

# Now when you map:
y_true_enc = df["query_label"].map({c:i for i,c in enumerate(le.classes_)}).to_numpy()
y_pred_enc = df["target_label"].map({c:i for i,c in enumerate(le.classes_)}).to_numpy()

# 3) map both columns to ints
y_true_enc = df["query_label"].map(cls2idx).to_numpy()
y_pred_enc = df["target_label"].map(cls2idx).to_numpy()

# 4) compute metrics
accuracy = accuracy_score(y_true_enc, y_pred_enc)
mcc      = matthews_corrcoef(y_true_enc, y_pred_enc)

# micro‐MCC over flattened one‐hots
n_classes  = len(class_list)
y_true_bin = label_binarize(y_true_enc, classes=range(n_classes))
y_pred_bin = label_binarize(y_pred_enc, classes=range(n_classes))
micro_mcc  = matthews_corrcoef(y_true_bin.ravel(), y_pred_bin.ravel())

# per-class report (dict form)
report_dict = classification_report(
    y_true_enc,
    y_pred_enc,
    labels       = range(n_classes),
    target_names = class_list,
    output_dict  = True,
    zero_division=0
)

# 5) print metrics
print(f"Accuracy : {accuracy:.4f}")
print(f"MCC      : {mcc:.4f}")
print(f"Micro‐MCC: {micro_mcc:.4f}\n")
print("Per-class metrics:\n")
print(classification_report(
    y_true_enc,
    y_pred_enc,
    labels       = range(n_classes),
    target_names = class_list,
    zero_division=0
))

plot_confusion_matrix(
  all_labels=y_true_enc,
  all_preds=y_pred_enc,
  label_encoder=le,
  output_path="confusion_matrix.png")

# ——— drop-in: save to JSON and CSV ———

numeric_metrics = {
    "Test_Accuracy": accuracy,
    "Test_MCC": mcc,
    "Test_Micro_MCC": micro_mcc
}

output = {
    "numeric_metrics": numeric_metrics,
    "classification_report": report_dict
}

out_path = Path("test_metrics.json")
out_path.write_text(json.dumps(output, indent=4))
print(f"Saved metrics JSON to {out_path}")

# save CSV
out_path = Path("best_all.csv")
best_all.to_csv("best_all.csv", index=False)
print(f"Saved CSV to {out_path}")

Accuracy : 0.9821
MCC      : 0.8479
Micro‐MCC: 0.9816

Per-class metrics:

                                                         precision    recall  f1-score   support

                                                  other       0.78      0.65      0.71        49
                                         Insulin family       1.00      1.00      1.00         2
                                   Phospholipase family       0.73      0.85      0.79        26
                        Flavin monoamine oxidase family       1.00      1.00      1.00         2
                                     Actinoporin family       1.00      1.00      1.00         4
                                       Conotoxin family       0.97      0.92      0.95       113
                  Venom metalloproteinase (M12B) family       1.00      0.89      0.94        19
                              Three-finger toxin family       1.00      1.00      1.00        36
    Natriuretic, Bradykinin potentiating peptide fa

In [38]:
df["target_label"].value_counts()

target_label
no hit    8609
Name: count, dtype: int64